In [5]:

from SimpleTokenizer import SimpleTokenizer
import re






with open ("sample.txt","r") as f:
    raw_text = f.read()

processed = re.split(r'[.,/:;? \s]',raw_text)

processed =[item.strip() for item in processed if item.strip()]

all_words = sorted(set(processed))

vocab = {text:enc for enc,text in enumerate(all_words)}

tok  = SimpleTokenizer(vocab)

enc_text = tok.encode(raw_text)
print(len(enc_text))
  



3665


In [6]:
enc_sample = enc_text[50:]
context_size = 4
for i in range (1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{context} ----> {desired}")

[943] ----> 1128
[943, 1128] ----> 680
[943, 1128, 680] ----> 1255
[943, 1128, 680, 1255] ----> 611


In [8]:
for i in range (1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    context_d = tok.decode(context) 
    desired_d = tok.decode([desired]) ##since desired is a word but decode by definition takes a list
    print(f"{context_d} ----> {desired_d}")

rather ----> thought
rather thought ----> it
rather thought it ----> would
rather thought it would ----> have


In [14]:
## our gpt dataloader is in essence taking in raw text and processing it till we get input_ids as an attribute to the loader and essentially creating labelled input and target data
import torch
from torch.utils.data import Dataset,DataLoader
import tiktoken

class GPTdataloader(Dataset):
    def __init__(self,txt,tok,max_length,stride):
        self.input_ids = []
        self.target_ids = []
        encoded_words = tok.encode(txt)
        for i  in range (1,len(encoded_words)-max_length,stride):
            inp = encoded_words[i:i+max_length]
            tar = encoded_words[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(inp))
            self.target_ids.append(torch.tensor(tar))
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]



In [18]:
## creating dataloader

def creat_dataloader_v1(txt,batch_size = 4,max_length = 256,stride = 128,shuffle = True,drop_last = True,num_workers=0):
    tokenizer=tiktoken.get_encoding("gpt2")
    dataset = GPTdataloader(txt,tokenizer,max_length,stride)
    dataloader = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
    return dataloader

In [19]:
with open("sample.txt","r") as f:
    raw = f.read()
dataloader = creat_dataloader_v1(raw,batch_size =1,max_length=4,stride = 1,shuffle = False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [20]:
## creaating embeddings. essentially we are not using the input_ids and toke ids for creating the embeddings directly. we randomise the weights and create the embedding weights of required dimensions and each token id gets assgined an embedding.
## it is the training part and the backpropagation that changes the embedding parameters 

vocab_size = 9 # just for the sake of assumption and showing the matrix structure
out_dims = 5

embedding_layer = torch.nn.Embedding(vocab_size,out_dims)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 1.6378,  0.9822, -0.3495,  1.0811,  1.3098],
        [-0.6899, -0.3503,  1.4619,  0.2982, -0.0820],
        [-2.2859, -1.9613,  2.6627,  3.4202, -0.2405],
        [ 1.0078,  0.9871,  0.4893, -0.7995,  0.7326],
        [ 2.1400, -0.2895, -0.7600,  0.4333, -1.0188],
        [ 2.2815,  1.0777,  0.0217, -0.5914,  0.7139],
        [ 2.1985,  1.9144,  1.8624,  1.2472, -0.5647],
        [-1.6777,  0.0525, -0.6171, -1.2767,  0.0177],
        [ 0.8959,  0.2140,  1.3503,  0.5129, -1.0656]], requires_grad=True)


In [21]:
## as seen above we have weights initialised for each of the 9 letter sin our assumed vocabualary. obviously the vocab size will be more and these random weights can be initialised to that approporiate size
# the problem is that right now the embeddings capture word significance, ie if we use just this table/layer for training then the words having similar meaning will have similar vectors. but the positional intuition for the model wont be there, hence we alse create positional embeddings
vocab_size = 50257
out_dims = 256
token_embedding_layer = torch.nn.Embedding(vocab_size,out_dims)
## we are using the data loader to generate instances to embedd
max_length  = 4
dataloader = creat_dataloader_v1(raw,batch_size=8,max_length=max_length,stride=max_length,shuffle=False)
data_iter=iter(dataloader)
inputs,targets = next(data_iter)
print(inputs)

tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [24]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [ ]:
# now for each input, the context_length is max_length and we need positional embeddings for that
context_length = max_length
pos_embedding_layer=torch.nn.Embedding(context_length,out_dims)
## now each of the 4 posistions in the context length has its own embeddnig and hence it is better for the attention layer to understand the embeddings